# Procesul de difuzie, pas cu pas

Procesul invers al unui model de difuzie, scris de mână cu Diffusers pe un U-Net care generează pisici. Zgomotul estimat la fiecare pas, estimarea imaginii curate, DDPM față de DDIM și procesul direct, verificat pe formula lui.

Codul din acest notebook vine de pe pagina [Procesul de difuzie, pas cu pas](https://inteligentartificiala.ro/laboratoare/difuzie-imagini-si-sunete/procesul-de-difuzie/) a laboratorului [Imagini și sunete cu modele de difuzie](https://inteligentartificiala.ro/laboratoare/difuzie-imagini-si-sunete/prezentare/), de pe [InteligentArtificiala.ro](https://inteligentartificiala.ro). Explicațiile fiecărui pas sunt pe pagină.

În Google Colab, codul rulează mult mai repede pe o placă grafică. Dacă nu este deja aleasă, se alege din meniul **Runtime**, opțiunea **Change runtime type**, o variantă cu GPU. Celulele se rulează în ordine, de sus în jos.

In [ ]:
%pip install -q -U diffusers transformers accelerate

## Programul de zgomot

Diffusers separă rețeaua de **metoda de eșantionare** (_scheduler_). Rețeaua, aici un `UNet2DModel`, primește o imagine cu zgomot și pasul $t$ și estimează zgomotul. Metoda de eșantionare păstrează programul de zgomot și calculează, din estimarea rețelei, imaginea de la pasul următor. Codul de mai jos încarcă doar metoda de eșantionare și afișează, pentru câțiva pași, coeficienții din formula procesului direct, $x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1 - \bar\alpha_t}\, \varepsilon$.

In [ ]:
from diffusers import DDPMScheduler

# Metoda de eșantionare DDPM, cu programul de zgomot folosit la antrenarea modelului.
scheduler = DDPMScheduler.from_pretrained("google/ddpm-cat-256")
print(f"beta_schedule = {scheduler.config.beta_schedule}, T = {scheduler.config.num_train_timesteps}")

for t in [0, 250, 500, 750, 999]:
    alpha_bar = scheduler.alphas_cumprod[t]  # ᾱ_t
    print(f"t = {t:3d}: {alpha_bar.sqrt():.3f} × imaginea + {(1 - alpha_bar).sqrt():.3f} × zgomotul")

## Procesul invers, scris de mână

Codul de mai jos pornește de la zgomot pur, $x_T$, și face 50 de pași cu metoda **DDIM**, a cărei regulă este descrisă pe pagina despre [generare](https://inteligentartificiala.ro/modele-de-difuzie/ce-sunt-modelele-de-difuzie/#generarea-de-la-zgomot-la-imagine). La câțiva pași, codul salvează două imagini într-o grilă: imaginea cu zgomot, $x_t$, și estimarea imaginii curate, $\hat x_0$, calculată de metoda de eșantionare din zgomotul estimat de rețea.

In [ ]:
import torch
from diffusers import DDIMScheduler, UNet2DModel
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
model_id = "google/ddpm-cat-256"
unet = UNet2DModel.from_pretrained(model_id).to(device)
scheduler = DDIMScheduler.from_pretrained(model_id)  # aceeași rețea, cu metoda de eșantionare DDIM
scheduler.set_timesteps(50)


def to_image(x):
    # Un tensor cu valori între -1 și 1 devine o imagine cu pixeli între 0 și 255.
    x = (x[0].clamp(-1, 1) + 1) / 2
    return Image.fromarray((x.permute(1, 2, 0).cpu().numpy() * 255).round().astype("uint8"))


generator = torch.Generator("cpu").manual_seed(0)
x = torch.randn((1, 3, 256, 256), generator=generator).to(device)  # x_T, zgomot pur
shown = [0, 10, 20, 30, 40, 49]  # pașii afișați în grilă
grid = Image.new("RGB", (len(shown) * 256, 2 * 256))
for i, t in enumerate(scheduler.timesteps):
    with torch.no_grad():
        noise_pred = unet(x, t).sample  # zgomotul estimat de rețea, ε̂
    step = scheduler.step(noise_pred, t, x)
    if i in shown:
        grid.paste(to_image(x), (256 * shown.index(i), 0))  # sus: x_t
        grid.paste(to_image(step.pred_original_sample), (256 * shown.index(i), 256))  # jos: x̂_0
    x = step.prev_sample  # x_{t-1}, cu ceva mai puțin zgomot

grid.save("pasi.png")
to_image(x).save("pisica.png")

In [ ]:
import IPython.display as ipd

ipd.display(ipd.Image("pasi.png"))
ipd.display(ipd.Image("pisica.png"))

## DDPM și DDIM

Aceeași rețea poate fi folosită cu metode de eșantionare diferite, fără nicio antrenare suplimentară. Codul de mai jos generează câte o pisică prin DDPM, cu toți cei 1.000 de pași, și prin DDIM, cu 50 de pași, pornind de la același zgomot, și măsoară cât durează fiecare. Pentru că pipeline-urile `DDPMPipeline` și `DDIMPipeline` fac aceiași pași ca mai sus, codul este mai scurt.

In [ ]:
import time

import torch
from diffusers import DDIMPipeline, DDIMScheduler, DDPMPipeline
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
ddpm = DDPMPipeline.from_pretrained("google/ddpm-cat-256").to(device)
# Aceeași rețea, cu metoda de eșantionare DDIM.
ddim = DDIMPipeline(unet=ddpm.unet, scheduler=DDIMScheduler.from_config(ddpm.scheduler.config))

grid = Image.new("RGB", (2 * 256, 256))
for column, (name, pipe, steps) in enumerate([("DDPM", ddpm, 1000), ("DDIM", ddim, 50)]):
    start = time.time()
    generator = torch.Generator("cpu").manual_seed(0)
    image = pipe(num_inference_steps=steps, generator=generator).images[0]
    print(f"{name}: {steps} de pași în {time.time() - start:.1f} secunde")
    grid.paste(image, (256 * column, 0))
grid.save("ddpm_ddim.png")

In [ ]:
import IPython.display as ipd

ipd.display(ipd.Image("ddpm_ddim.png"))

## Procesul direct

Procesul direct nu are nevoie de rețea: este doar formula programului de zgomot. Metoda `add_noise` o aplică direct pentru orice pas $t$, fără pașii de dinainte, exact ca la antrenare. Codul de mai jos adaugă zgomot pisicii generate mai sus, salvată în `pisica.png`, și compară rezultatul cu formula scrisă de mână.

In [ ]:
import numpy as np
import torch
from diffusers import DDPMScheduler
from PIL import Image

scheduler = DDPMScheduler.from_pretrained("google/ddpm-cat-256")
# Pisica generată mai sus, cu pixelii aduși între -1 și 1.
x0 = torch.from_numpy(np.array(Image.open("pisica.png"))).permute(2, 0, 1)[None].float() / 127.5 - 1
noise = torch.randn(x0.shape, generator=torch.Generator().manual_seed(0))  # ε

grid = Image.new("RGB", (5 * 256, 256))
for column, t in enumerate([0, 250, 500, 750, 999]):
    x_t = scheduler.add_noise(x0, noise, torch.tensor([t]))
    alpha_bar = scheduler.alphas_cumprod[t]
    by_hand = alpha_bar.sqrt() * x0 + (1 - alpha_bar).sqrt() * noise  # x_t = √ᾱ_t · x_0 + √(1 − ᾱ_t) · ε
    print(f"t = {t:3d}: diferența maximă față de formula scrisă de mână este {(x_t - by_hand).abs().max():.1e}")
    pixels = ((x_t[0].clamp(-1, 1) + 1) * 127.5).permute(1, 2, 0).round().byte().numpy()
    grid.paste(Image.fromarray(pixels), (256 * column, 0))
grid.save("procesul_direct.png")

In [ ]:
import IPython.display as ipd

ipd.display(ipd.Image("procesul_direct.png"))